# Eishkom Stage 1: Colab Runner (Vertex AI Colab Enterprise & Free Colab)
Open this notebook in **Vertex AI Colab Enterprise** (funded by your **$60 Google Cloud credits**) or free Google Colab.
Run the cells from top to bottom, **one at a time**, and inspect what each prints.

### Hardware & Budget Recommendations:
- **Vertex AI Colab Enterprise (Recommended):** Choose an **NVIDIA L4 GPU (24GB VRAM)** with **Spot VM pricing** (~$0.32/hr).
  - Zero random browser disconnects.
  - Large 24GB VRAM allows training **koeberg-stage-1 (27B)** in ~5.5 hrs (~$1.76) and **kendal-stage-1 (4B)** in ~1 hr (~$0.32).
  - *Quota Note:* Before first run, request NVIDIA L4 GPU quota = 1 in GCP Console (IAM & Admin > Quotas).
- **Free Google Colab ($0):** Standard NVIDIA T4 (15GB usable VRAM). Good for testing 4B and 2B models.

**Before you start:** upload `sa-tutor-kit` (including `workspace/`) to Google Drive (My Drive).

In [ ]:
# 1. Is there a GPU?
!nvidia-smi

## 2. Connect Google Drive and go to the kit folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
KIT = '/content/drive/MyDrive/sa-tutor-kit'          # <- change if you put it somewhere else
os.environ['SA_WORKSPACE'] = KIT + '/workspace'       # all scripts read/write here
%cd {KIT}
!ls

## 3. Install the tools (about 3-5 minutes)
If Unsloth's own Qwen3.5 notebook uses a different install line, **use theirs** (they update it often).

In [ ]:
!pip install -q unsloth
!pip install -q sympy pypdf matplotlib

## 4. Hugging Face login (only needed for models that ask you to accept a licence, e.g. Gemma)

In [ ]:
from huggingface_hub import login
login()

## 5. AUDITION base models (before any training)
Each run asks candidate models the same held-out exam.
**Critical Rule:** Only pass unquantized base Hugging Face repositories here. **Never** audition or train on a `-GGUF` file directly.

In [ ]:
MODELS = {
    "base-qwen35-4b": "unsloth/Qwen3.5-4B",
    "base-qwen38-4b-distill": "empero-ai/Qwen3.8-4B-Distill",
    "base-qwen38-2b-distill": "empero-ai/Qwen3.8-2B-Distill",
    # "base-qwen38-27b": "unsloth/Qwen3.8-27B",  # Needs L4 GPU (24GB VRAM)
    # "base-liquid-2b": "LiquidAI/LFM3-VL-2B",
}
for name, model in MODELS.items():
    style = "gemma" if "gemma" in model.lower() else "qwen"
    !python eval/run_eval.py --name {name} --backend hf --model {model} --load-4bit --think-style {style} --tags core --max-new-tokens 800
    !python eval/score_eval.py --name {name}


In [ ]:
!python eval/report_eval.py --runs base-qwen35-4b --out audition

## 6. TRAIN Stage 1 Models with Unsloth QLoRA
Fine-tune base models using Unsloth 4-bit QLoRA. Step 6a is a free dry-run that checks rendering and tokenizer alignment.

### Eishkom Stage 1 Lineup:
1. **Koeberg (27B)**: `unsloth/Qwen3.8-27B` on L4 Spot VM (~5.5 hrs, ~$1.76)
2. **Kendal (4B)**: `unsloth/Qwen3.5-4B` on L4 Spot VM (~1.0 hr, ~$0.32)
3. **Gariep (2B)**: `empero-ai/Qwen3.8-2B-Distill` on L4 Spot VM (~35 mins, ~$0.19)
*(Tip: Train Koeberg first, use it to solve hard questions, and pass its reasoning down into Kendal and Gariep!)*

In [ ]:
ROUND = "r1"                                           # name of your build folder under workspace/build
# Choose the base model for this run:
MODEL = "unsloth/Qwen3.5-4B"                           # Kendal Desktop (4B)
# MODEL = "unsloth/Qwen3.8-27B"                         # Koeberg Flagship Cloud (27B, requires L4 GPU)
# MODEL = "empero-ai/Qwen3.8-2B-Distill"                # Gariep Budget Tutor (2B)
!python training/train_sft.py --model {MODEL} --data workspace/build/{ROUND} --load-4bit --dry-run --show 2


6b. **The "can it memorise?" test** (about 5-10 minutes). 20 samples, 15 passes. The loss at the end should be close to zero.
If it is not, something is wrong with the data or the template: do NOT start the long run.

In [ ]:
!python training/train_sft.py --model {MODEL} --data workspace/build/{ROUND} --out workspace/runs/overfit --limit 20 --epochs 15 --show 0

6c. **The real training run.** Checkpoints are saved to Google Drive after each epoch.
If disconnected, re-run with `--resume`. On an L4 Spot VM, 4B takes ~1 hour (~$0.32); 27B takes ~5.5 hours (~$1.76).

In [ ]:
!python training/train_sft.py --model {MODEL} --data workspace/build/{ROUND} --out workspace/runs/{ROUND} --load-4bit --epochs 3 --show 0


## 7. EVALUATE the trained model (one command: answer, mark, compare, report)
`--compare` lists earlier runs (oldest first). `--ceiling` is the teacher's run on the same exam. Read `decision.md` and `failures.md`.

In [ ]:
!python eval/evaluate.py --name {ROUND} --backend hf --model {MODEL} --adapter workspace/runs/{ROUND}/adapter --compare base-qwen35-4b --ceiling teacher --analyze

## 8. More samples for consistency (5 answers per question at temperature 0.7)
A model that is right 5 times out of 5 is far more trustworthy than one that is right 3 times out of 5.

In [ ]:
!python eval/run_eval.py --name {ROUND}-n5 --backend hf --model {MODEL} --adapter workspace/runs/{ROUND}/adapter --tags core --n 5 --temperature 0.7
!python eval/score_eval.py --name {ROUND}-n5

## 9. EXPORT GGUF files with clean Eishkom branding
Export deployable GGUF files for offline devices and Cloud Run.
**Clean Branding Rule:** Public model names must not show parameter or quant letters (e.g. `kendal-stage-1` and `kendal-flash-stage-1`).
From a single 4B training run, export both Q6_K and Q4_K_M in minutes!

In [ ]:
# Export Kendal Desktop (Q6_K) and Kendal Flash Mobile (Q4_K_M) from 1 run:
!python training/export_gguf.py --adapter workspace/runs/{ROUND}/adapter --out workspace/exports/kendal-stage-1 --quants q6_k q4_k_m

# If exporting Koeberg (27B), export to UD-Q4_K_XL for Cloud Run:
# !python training/export_gguf.py --adapter workspace/runs/r1-koeberg/adapter --out workspace/exports/koeberg-stage-1 --quants ud-q4_k_xl

# If exporting Gariep (2B), export to Q6_K and Q8_0:
# !python training/export_gguf.py --adapter workspace/runs/r1-gariep/adapter --out workspace/exports/gariep-stage-1 --quants q6_k q8_0


## 10. Preference training (DPO), later
After SFT passes the content checks but habits remain (leaks, language drift): build pairs with `training/make_dpo_pairs.py`
(GUIDE.md phase 9), then train with **Unsloth's official DPO notebook**, pointing its dataset cell at your pairs file
(`workspace/build/dpo_<round>.jsonl`, standard prompt / chosen / rejected format).

## 11. Google Cloud Run Deployment: Host koeberg-stage-1 with Scale-to-Zero
To host your 27B flagship model in the cloud without burning your credits when idle, deploy to **Google Cloud Run** with a Serverless NVIDIA L4 GPU.
- `--min-instances 0`: Scales down to 0 when idle ($0.00/hr).
- Boots on query: ~$0.015/minute during active chat.
- Covered entirely by your $60 promotional credits!

In [ ]:
# Cloud Run Deployment Command (run from Cloud Shell or local gcloud):
# gcloud run deploy koeberg-stage-1 \
#   --image gcr.io/$PROJECT_ID/koeberg-stage-1:latest \
#   --gpu 1 \
#   --gpu-type nvidia-l4 \
#   --cpu 8 \
#   --memory 32Gi \
#   --min-instances 0 \
#   --max-instances 1 \
#   --region us-central1
